---
title: "Download Module: Downloading Raw Data from CDSAPI"
engine: jupyter
---

## download

> This module downloads the raw data from CDS and saves it in the local directory

<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

We use a similar approach to the one in the tutorial to download the data
to local storage.

The background functionality in this module involves downloading the
bounding box of a region of interest, and sending that to the
CDS API query. As such, we define two helper functions to
fetch the OCHA/HDX shapefiles for a geographic region, and
another to create the bounding box from the files.

In [0]:
#| echo: false
#| output: asis
show_doc(fetch_GADM)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/download.py#L58){target="_blank" style="float:right; font-size:smaller"}

### fetch_GADM

>      fetch_GADM
>                  (url:str='https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm4
>                  1_MDG.gpkg', output_file:str='gadm41_MDG.gpkg')

*Fetch the GADM bounding box for geographic region*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| url | str | https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_MDG.gpkg | URL to fetch the GADM data for Madagascar |
| output_file | str | gadm41_MDG.gpkg | file path to save the GADM data |
| **Returns** | **str** |  |  |

In [0]:
#| echo: false
#| output: asis
show_doc(create_bounding_box)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/download.py#L79){target="_blank" style="float:right; font-size:smaller"}

### create_bounding_box

>      create_bounding_box (zip_url_or_path:str, buffer_km:float=50,
>                           round_to:int=1)

*Create a bounding box from OCHA/HDX shapefile data with a buffer.*

|    | **Type** | **Default** | **Details** |
| -- | -------- | ----------- | ----------- |
| zip_url_or_path | str |  | URL or local path to the zipped shapefile. |
| buffer_km | float | 50 | Buffer distance in kilometers to expand the bounding box. |
| round_to | int | 1 | Number of decimal places to round the bounding box coordinates. |
| **Returns** | **list** |  | **Bounding box in the CDS API area format [North, West, South, East]** |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def create_bounding_box(
    zip_url_or_path: str, # URL or local path to the zipped shapefile.
    buffer_km: float = 50, # Buffer distance in kilometers to expand the bounding box.
    round_to: int = 1 # Number of decimal places to round the bounding box coordinates.
) -> list: # Bounding box in the CDS API area format [North, West, South, East]
    '''
    Create a bounding box from OCHA/HDX shapefile data with a buffer.
    '''
    with tempfile.TemporaryDirectory() as tmpdir:
        # Download if it's a URL
        if zip_url_or_path.startswith("http"):
            response = requests.get(zip_url_or_path)
            zip_path = os.path.join(tmpdir, "ocha_data.zip")
            with open(zip_path, "wb") as f:
                f.write(response.content)
        else:
            zip_path = zip_url_or_path

        # Unzip
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(tmpdir)

        # Find the .shp file
        shp_files = list(Path(tmpdir).rglob("*.shp"))
        if not shp_files:
            raise FileNotFoundError("No shapefile (.shp) found in the extracted archive.")
        shp_path = str(shp_files[0])  # Use first found .shp

        # Read shapefile
        shape = gpd.read_file(shp_path)

        # Reproject to projected CRS (you may want to detect the correct UTM zone)
        shape_proj = shape.to_crs(epsg=32738)

        # Apply buffer
        buffered = shape_proj.geometry.buffer(buffer_km * 1000)

        # Convert back to geographic coordinates
        buffered_geo = gpd.GeoSeries(buffered, crs=shape_proj.crs).to_crs(epsg=4326)

        # Get bounding box
        bounds = buffered_geo.total_bounds  # [min_x, min_y, max_x, max_y]
        bbox = [
            round(bounds[3], round_to),  # North
            round(bounds[0], round_to),  # West
            round(bounds[1], round_to),  # South
            round(bounds[2], round_to)   # East
        ]

        return bbox

The primary function to download the data from CDSAPI is defined below.

In [0]:
#| echo: false
#| output: asis
show_doc(download_raw_era5)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/download.py#L132){target="_blank" style="float:right; font-size:smaller"}

### download_raw_era5

>      download_raw_era5 (cfg:omegaconf.dictconfig.DictConfig)

*Send the query to the API and download the data*

|    | **Type** | **Details** |
| -- | -------- | ----------- |
| cfg | DictConfig | hydra configuration file |
| **Returns** | **None** |  |

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
def download_raw_era5(
        cfg: DictConfig  # hydra configuration file
    )->None:
    '''
    Send the query to the API and download the data
    '''

    # parse the cfg
    testing = cfg.development_mode # for testing
    output_dir = here("data/input") # output directory
    
    geography = cfg.query.geography

    target = os.path.join(_expand_path(output_dir), "{}_{}_{}.nc".format(geography, cfg.query['year'], cfg.query['month']))
    
    client = cdsapi.Client()
    
    query = _validate_query(cfg.query)

    dataset = cfg.dataset
    # to make sure the query is valid at the end
    del query['geography']
    
    # Send the query to the client
    if not testing:
        bounds = create_bounding_box(cfg.geographies[geography]['shapefile'])
        query['area'] = bounds
        client.retrieve(dataset, query).download(target)

        print("Downloaded file to: {}".format(target))
    else:
        print(f"Testing mode. Not downloading data. Query is {query}")

    print("Done")

## Tests and Main

Here we define some tests and the main function that will be used to download the data.

In [ ]:
#| eval: false
from hydra import initialize, compose
from omegaconf import OmegaConf

# unfortunately, we have to use the initialize function to load the config file
# this is because the @hydra decorator does not work with Notebooks very well
# this is a known issue with Hydra: https://gist.github.com/bdsaglam/586704a98336a0cf0a65a6e7c247d248
# 
# just use the relative path from the notebook to the config dir
try:
    with initialize(version_base=None, config_path="../conf"):
        cfg = compose(config_name='config.yaml')
except Exception as e:
    print(f"Error initializing Hydra: {e}")
    with initialize(version_base=None, config_path="conf"):
        cfg = compose(config_name='config.yaml')

cfg.development_mode = False
cfg.query['year'] = 2017
cfg.query['month'] = 11
#cfg.query['day'] = 1
#cfg.query['time'] = "00:00"
cfg.query['geography'] = "nepal"
download_raw_era5(cfg)

In [0]:
#| echo: false
#| output: asis
show_doc(main)

---

[source](https://github.com/TinasheMTapera/era5_sandbox/blob/main/era5_sandbox/aggregate.py#L302){target="_blank" style="float:right; font-size:smaller"}

### main

>      main (cfg:omegaconf.dictconfig.DictConfig)

In [ ]:
#| code-fold: show
#| code-summary: "Exported source"
#| exports: #
@hydra.main(config_path="../../conf", config_name="config", version_base=None)
def main(cfg: DictConfig) -> None:
    download_raw_era5(cfg=cfg)
    # better approach would be to have the function only use the specific arguments of the config